# Spotify Songs Genre Segmentation & Recommendation System

An unsupervised ML pipeline that clusters ~32,000 Spotify tracks by audio features using K-Means and PCA, then builds a content-based song recommendation engine using Euclidean distance within clusters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import euclidean_distances

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

## 1. Data Loading & Pre-processing

In [ ]:
df = pd.read_csv('spotify dataset.csv')
print(f"Raw dataset shape: {df.shape}")

# Drop missing values
df = df.dropna()
df = df.reset_index(drop=True)   # Critical: ensures iloc indexing works correctly later
print(f"Shape after dropping nulls: {df.shape}")
print(f"\nGenres in dataset: {df['playlist_genre'].unique()}")
print(f"Unique genres: {df['playlist_genre'].nunique()}")

In [ ]:
# Select Spotify audio features for clustering
features = ['danceability', 'energy', 'key', 'loudness', 'mode',
            'speechiness', 'acousticness', 'instrumentalness',
            'liveness', 'valence', 'tempo', 'duration_ms']

X = df[features].copy()

# Scale features — essential since duration_ms (~200000) dwarfs danceability (0-1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Features used: {features}")

## 2. Exploratory Data Analysis

In [ ]:
# Distribution of all audio features
X.hist(bins=30, figsize=(15, 12), color='teal')
plt.suptitle('Distribution of Spotify Audio Features', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# Song count by genre
plt.figure(figsize=(10, 5))
sns.countplot(y='playlist_genre', data=df,
              order=df['playlist_genre'].value_counts().index, palette='magma')
plt.title('Number of Songs per Playlist Genre')
plt.xlabel('Count')
plt.show()

In [ ]:
# Danceability and Energy by genre — key differentiating features
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(x='playlist_genre', y='danceability', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Danceability by Genre')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(x='playlist_genre', y='energy', data=df, ax=axes[1], palette='Set3')
axes[1].set_title('Energy by Genre')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 3. Correlation Matrix

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(X.corr(), annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Correlation Matrix of Audio Features')
plt.show()

# Key insight: loudness-energy are strongly correlated; acousticness-energy are negatively correlated
print("Key correlations:")
corr = X.corr()
print(f"  loudness ↔ energy:       {corr.loc['loudness','energy']:.2f}")
print(f"  acousticness ↔ energy:   {corr.loc['acousticness','energy']:.2f}")
print(f"  danceability ↔ valence:  {corr.loc['danceability','valence']:.2f}")

## 4. Dimensionality Reduction with PCA

In [ ]:
# Reduce to 2D for visualization
pca_2d = PCA(n_components=2, random_state=42)
X_pca  = pca_2d.fit_transform(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

explained_2d = pca_2d.explained_variance_ratio_.sum() * 100
print(f"Variance explained by 2 components: {explained_2d:.1f}%")

# Check how many components needed for 80% variance
pca_full = PCA(random_state=42).fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
n_80   = np.argmax(cumvar >= 80) + 1
print(f"Components needed for 80% variance: {n_80}")
print(f"Note: 2D PCA is used for visualization only; clustering uses all 12 scaled features.")

# Explained variance plot
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(cumvar)+1), cumvar, marker='o', color='teal')
plt.axhline(80, color='red', linestyle='--', label='80% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance (%)')
plt.title('PCA — Cumulative Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visualize true genre labels in PCA space
plt.figure(figsize=(12, 8))
sns.scatterplot(x='PCA1', y='PCA2', hue='playlist_genre', data=df,
                palette='tab10', alpha=0.5, s=12)
plt.title(f'Songs by Actual Genre in PCA Space ({explained_2d:.1f}% variance)')
plt.legend(title='Genre', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5. Finding Optimal K — Elbow Method + Silhouette Score

In [ ]:
# Use a subset for speed; elbow shape is consistent with full data
subset_size = min(10000, len(X_scaled))
X_sub = X_scaled[:subset_size]

wcss       = []
sil_scores = []
k_range    = range(2, 11)

print("Computing Elbow and Silhouette scores...")
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sub)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_sub, labels))
    print(f"  k={k}: WCSS={km.inertia_:,.0f}  Silhouette={sil_scores[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, wcss, marker='o', linestyle='--', color='teal')
axes[0].set_title('Elbow Method — WCSS vs k')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('WCSS')

axes[1].plot(k_range, sil_scores, marker='o', linestyle='--', color='coral')
axes[1].set_title('Silhouette Score vs k (Higher = Better)')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')

plt.suptitle('Optimal K Selection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best_k_sil = list(k_range)[sil_scores.index(max(sil_scores))]
print(f"\nBest k by Silhouette Score: {best_k_sil} (score: {max(sil_scores):.4f})")
print(f"Chosen k=6 — matches the 6 actual genres in the dataset.")

## 6. K-Means Clustering (k=6)

In [ ]:
k = 6
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Silhouette score on full dataset
sil_full = silhouette_score(X_scaled, df['Cluster'])
print(f"Silhouette Score (full dataset, k=6): {sil_full:.4f}")
print("Interpretation: scores 0.1–0.25 are typical for music data due to genre overlap.")

# Cluster sizes
print(f"\nCluster sizes:")
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# Visualize ML clusters in PCA space
plt.figure(figsize=(12, 8))
sns.scatterplot(x='PCA1', y='PCA2', hue='Cluster', data=df,
                palette='Set1', alpha=0.5, s=12)
plt.title(f'K-Means Clusters in PCA Space (k={k})')
plt.legend(title='ML Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-tab: ML Clusters vs Actual Genres
crosstab = pd.crosstab(df['playlist_genre'], df['Cluster'])
plt.figure(figsize=(10, 6))
sns.heatmap(crosstab, cmap='Blues', annot=True, fmt="d")
plt.title('Actual Genre vs K-Means Cluster Assignment')
plt.ylabel('Actual Genre')
plt.xlabel('K-Means Cluster')
plt.tight_layout()
plt.show()

print("\nDominant genre per cluster:")
for c in range(k):
    cluster_genres = df[df['Cluster'] == c]['playlist_genre'].value_counts()
    dominant = cluster_genres.index[0]
    pct = cluster_genres.iloc[0] / cluster_genres.sum() * 100
    print(f"  Cluster {c}: {dominant} ({pct:.1f}%)")

## 7. Song Recommendation Engine

In [ ]:
def recommend_songs(song_name, df, X_scaled, num_recommendations=5):
    """
    Content-based recommendation using K-Means cluster assignment 
    and Euclidean distance on scaled audio features.
    """
    matches = df[df['track_name'].str.lower() == song_name.lower()]

    if matches.empty:
        return f"Song '{song_name}' not found in dataset."

    # Use first match
    song_idx     = matches.index[0]
    song_cluster = df.loc[song_idx, 'Cluster']
    song_vec     = X_scaled[song_idx].reshape(1, -1)

    # Get all songs in the same cluster (excluding the query song)
    cluster_idx      = df[(df['Cluster'] == song_cluster) & (df.index != song_idx)].index
    cluster_features = X_scaled[cluster_idx]

    # Find most similar songs by Euclidean distance
    distances   = euclidean_distances(song_vec, cluster_features)[0]
    nearest_idx = cluster_idx[np.argsort(distances)[:num_recommendations]]

    result = df.loc[nearest_idx, ['track_name', 'track_artist', 'playlist_genre']].copy()
    result['similarity_distance'] = np.sort(distances)[:num_recommendations].round(3)
    return result.reset_index(drop=True)

# Demo
for song in ['bad guy', 'Shape of You', 'Blinding Lights']:
    print(f"\n--- Recommendations for '{song}' ---")
    print(recommend_songs(song, df, X_scaled, num_recommendations=5))